# 07 â€” Indexation dense LanceDB et lexicale BM25

Construit depuis zÃ©ro les index ENSEA, Ã©conomique et Facebook utilisÃ©s par l'application.

> VÃ©rifiez toujours les drapeaux `RUN_*` avant l'exÃ©cution. Pour cette reconstruction propre, les traitements locaux nÃ©cessaires des Ã©tapes 02 Ã  07 sont dÃ©jÃ  activÃ©s ; les collectes rÃ©seau et Ã©valuations GPU facultatives restent dÃ©sactivÃ©es.


In [1]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('DonnÃ©es actives :', PROJECT_ROOT / 'data')


Projet : <PROJECT_ROOT>
Python : <PROJECT_ROOT>\.venv\Scripts\python.exe
DonnÃ©es actives : <PROJECT_ROOT>\data


In [2]:
import json
import numpy as np
import pandas as pd
from chunker import Chunk
from config import load_config
from vectorstore import VectorStoreManager

config = load_config(PROJECT_ROOT / 'config.yaml')
chunk_path = config.project.data_path / 'chunks' / 'chunks.jsonl'
chunks = [Chunk(**json.loads(line)) for line in chunk_path.read_text(encoding='utf-8').splitlines() if line.strip()]
economic_path = config.local_documents.chunks_full_path
economic_chunks = [Chunk(**json.loads(line)) for line in economic_path.read_text(encoding='utf-8').splitlines() if line.strip()]
facebook_path = config.facebook_scraping.chunks_candidate_full_path
facebook_chunks = [Chunk(**json.loads(line)) for line in facebook_path.read_text(encoding='utf-8').splitlines() if line.strip()]
print('Chunks ENSEA :', len(chunks), '| Ã©conomiques :', len(economic_chunks), '| Facebook :', len(facebook_chunks))


Chunks ENSEA : 4719 | Ã©conomiques : 52486 | Facebook : 293


In [3]:
RUN_INDEXING = True
embedding_path = config.project.data_path / 'embeddings' / 'document_vectors.npy'
economic_embedding_path = config.local_documents.vectors_full_path
facebook_embedding_path = config.project.data_path / 'facebook' / 'vectors.npy'
if RUN_INDEXING:
    if not all(path.exists() for path in (embedding_path, economic_embedding_path, facebook_embedding_path)):
        raise FileNotFoundError('ExÃ©cutez le notebook 06_embeddings_bge_m3 avant celui-ci.')
    vectors = np.load(embedding_path)
    economic_vectors = np.load(economic_embedding_path)
    facebook_vectors = np.load(facebook_embedding_path)
    assert len(vectors) == len(chunks)
    assert len(economic_vectors) == len(economic_chunks)
    assert len(facebook_vectors) == len(facebook_chunks)

    production_store = VectorStoreManager(config)
    production_store.index_all(chunks, vectors)

    economic_config = config.model_copy(deep=True)
    economic_config.vectorstore.uri = config.local_documents.vectorstore_uri
    economic_config.vectorstore.table_name = config.local_documents.vectorstore_table
    economic_config.vectorstore.bm25_index_path = config.local_documents.bm25_index_path
    economic_store = VectorStoreManager(economic_config)
    economic_store.index_all(economic_chunks, economic_vectors)

    facebook_config = config.model_copy(deep=True)
    facebook_config.vectorstore.uri = config.facebook_scraping.vectorstore_candidate_uri
    facebook_config.vectorstore.table_name = config.facebook_scraping.vectorstore_candidate_table
    facebook_config.vectorstore.bm25_index_path = config.facebook_scraping.bm25_candidate_path
    facebook_store = VectorStoreManager(facebook_config)
    facebook_store.index_all(facebook_chunks, facebook_vectors)
else:
    raise RuntimeError('Lâ€™indexation est requise pour poursuivre le parcours.')


In [4]:
production_store.load_sparse()
economic_store.load_sparse()
facebook_store.load_sparse()
display(pd.Series({
    'vecteurs_ENSEA': production_store.dense.count(),
    'ENSEA_BM25': len(production_store.sparse.chunk_ids),
    'vecteurs_Ã©conomiques': economic_store.dense.count(),
    'Ã©conomie_BM25': len(economic_store.sparse.chunk_ids),
    'vecteurs_Facebook': facebook_store.dense.count(),
    'publications_BM25': len(facebook_store.sparse.chunk_ids),
}).to_frame('valeur'))
dense_example = production_store.dense.metadata_rows()[:3]
display(pd.DataFrame(dense_example)[['chunk_id', 'title', 'section', 'url']])


,valeur
vecteurs_ENSEA,4719
ENSEA_BM25,4719
vecteurs_Ã©conomiques,52486
Ã©conomie_BM25,52486
vecteurs_Facebook,293
publications_BM25,293


,chunk_id,title,section,url
0,0034fab77f8f62d2_7c002bef9d9cd254_1f68b141790b...,Frequently Asked Questions - ENSEA,Frequently Asked Questions - ENSEA,https://ensea.ed.ci/frequently-asked-questions/
1,0034fab77f8f62d2_7c002bef9d9cd254_ad9ff9b564eb...,Frequently Asked Questions - ENSEA,Frequently Asked Questions - ENSEA,https://ensea.ed.ci/frequently-asked-questions/
2,0034fab77f8f62d2_7c002bef9d9cd254_374e0a155211...,Frequently Asked Questions - ENSEA,Frequently Asked Questions - ENSEA,https://ensea.ed.ci/frequently-asked-questions/


## Pourquoi trois index ?

Chaque source possÃ¨de son couple LanceDB + BM25. La sÃ©paration protÃ¨ge les rÃ©ponses ENSEA du bruit potentiel des longs rapports Ã©conomiques et des publications sociales. Le routeur choisit d'abord l'index pertinent ; Ã  l'intÃ©rieur de cet index, LanceDB retrouve la proximitÃ© sÃ©mantique et BM25 privilÃ©gie les mots exacts, sigles et noms propres. Pour les tableaux, un troisiÃ¨me canal BM25 regroupe les cellules par ligne statistique afin de choisir ensuite la pÃ©riode exacte.


## SchÃ©ma d'index et contrÃ´les de reconstruction

LanceDB stocke le vecteur et les facettes structurÃ©es. BM25 indexe le titre, la section, le texte,
l'indicateur, la pÃ©riode, la valeur, l'unitÃ©, le territoire et le type de mesure. Les faits calculÃ©s sont indexÃ©s comme des preuves traÃ§ables. Le notebook vÃ©rifie ici que les deux index contiennent
exactement les mÃªmes chunks et que le nouveau schÃ©ma est actif.

In [5]:
stores = {
    'institutional': production_store,
    'economic': economic_store,
    'facebook': facebook_store,
}
schema_rows = []
for route, store in stores.items():
    metadata = store.dense.metadata_rows()
    content_counts = pd.Series([row.get('content_type', 'text') for row in metadata]).value_counts()
    schema_rows.append({
        'route': route,
        'schema_structurÃ©': store.dense.schema_is_current(),
        'dense': store.dense.count(),
        'bm25': len(store.sparse.chunk_ids),
        'text': int(content_counts.get('text', 0)),
        'table_fact': int(content_counts.get('table_fact', 0)),
        'derived_fact': int(content_counts.get('derived_fact', 0)),
    })
schema_df = pd.DataFrame(schema_rows)
display(schema_df)
assert schema_df['schema_structurÃ©'].all(), 'Un index LanceDB utilise encore lâ€™ancien schÃ©ma.'
assert schema_df['dense'].eq(schema_df['bm25']).all(), 'Dense et BM25 ne sont pas alignÃ©s.'

economic_metadata = economic_store.dense.metadata_rows()
import unicodedata
def normalized_label(value):
    normalized = unicodedata.normalize('NFKD', str(value).casefold())
    return ''.join(char for char in normalized if not unicodedata.combining(char))
critical_index_rows = pd.DataFrame([
    row for row in economic_metadata
    if row.get('content_type') == 'table_fact'
    and str(row.get('period')) == '2025'
    and normalized_label(row.get('indicator', '')) in {'population homme', 'rapport de masculinite'}
])
display(critical_index_rows[['indicator', 'period', 'value', 'unit', 'page', 'url']])
assert {'17103895', '109.3'} <= {str(value) for value in critical_index_rows['normalized_value']}

derived_index_rows = pd.DataFrame([
    row for row in economic_metadata if row.get('content_type') == 'derived_fact'
])
display(derived_index_rows[['indicator', 'period', 'value', 'unit', 'page', 'territory', 'measure_type', 'source_method']].head(30))
assert not derived_index_rows.empty, 'Les faits calculÃ©s ne sont pas prÃ©sents dans LanceDB.'
assert {'derived_annual_sum', 'derived_quarter_change'} <= set(derived_index_rows['source_method'])

,route,schema_structurÃ©,dense,bm25,text,table_fact,derived_fact
0,institutional,True,4719,4719,4719,0,0
1,economic,True,52486,52486,3657,48573,256
2,facebook,True,293,293,293,0,0


,indicator,period,value,unit,page,url
0,Population homme,2025,17 103 895,Nombre,10,/api/source-files/ANSTAT/Statistiques_Generale...
1,Rapport de masculinité,2025,"109,3",Nombre,10,/api/source-files/ANSTAT/Statistiques_Generale...


,indicator,period,value,unit,page,territory,measure_type,source_method
0,Total annuel de PRIMAIRE,2025,"8276,4",milliards de FCFA,14,Cote d'Ivoire,CVS | volume | prix chaines de 2015,derived_annual_sum
1,Variation de PRIMAIRE entre T1 2025 et T2 2025,T1 2025 à T2 2025,"4,9",%,14,Cote d'Ivoire,CVS | volume | prix chaines de 2015,derived_quarter_change
2,Variation de PRIMAIRE entre T2 2025 et T3 2025,T2 2025 à T3 2025,"-0,6",%,14,Cote d'Ivoire,CVS | volume | prix chaines de 2015,derived_quarter_change
3,Variation de PRIMAIRE entre T3 2025 et T4 2025,T3 2025 à T4 2025,"0,1",%,14,Cote d'Ivoire,CVS | volume | prix chaines de 2015,derived_quarter_change
4,Total annuel de INDUSTRIES AGRO-ALIMENTAIRES,2025,"2397,9",milliards de FCFA,14,Cote d'Ivoire,CVS | volume | prix chaines de 2015,derived_annual_sum
5,Variation de INDUSTRIES AGRO-ALIMENTAIRES entr...,T1 2025 à T2 2025,"2,6",%,14,Cote d'Ivoire,CVS | volume | prix chaines de 2015,derived_quarter_change
6,Variation de INDUSTRIES AGRO-ALIMENTAIRES entr...,T2 2025 à T3 2025,"3,1",%,14,Cote d'Ivoire,CVS | volume | prix chaines de 2015,derived_quarter_change
7,Variation de INDUSTRIES AGRO-ALIMENTAIRES entr...,T3 2025 à T4 2025,"1,4",%,14,Cote d'Ivoire,CVS | volume | prix chaines de 2015,derived_quarter_change
8,Total annuel de INDUSTRIES PETROLIERES,2025,"162,2",milliards de FCFA,14,Cote d'Ivoire,CVS | volume | prix chaines de 2015,derived_annual_sum
9,Variation de INDUSTRIES PETROLIERES entre T1 2...,T1 2025 à T2 2025,"11,5",%,14,Cote d'Ivoire,CVS | volume | prix chaines de 2015,derived_quarter_change
